# 第 13 章: 主成分分析の探索と可視化

累積寄与率と、第 1・第 2 主成分の散布図を確かめる。

Polyglot Notebooks（.NET Interactive）は 2026 年に廃止された。この Notebook は `Microsoft.dotnet-interactive` 1.0.712001 と Plotly.NET.Interactive 5.0.0 で動作を確かめている。
先に `dotnet build` で `apps/fsharp/` のライブラリをビルドしておく。

In [ ]:
#r "nuget: FSharp.Data, 8.2.0"
#r "nuget: Plotly.NET, 5.1.0"
#r "nuget: Plotly.NET.Interactive, 5.0.0"
#r "../src/MachineLearning/bin/Debug/net10.0/MachineLearning.dll"

In [ ]:
open System.IO
open Plotly.NET
open MachineLearning.Dataset
open MachineLearning.Chapter13.BostonStandardized
open MachineLearning.Chapter13.Pca

let bostonCsv = Path.Combine(dataDir (), "Boston.csv")
// 散布図の色分けに CRIME を使うので、標準化の前のデータも残しておく
let rows = loadBoston bostonCsv
let table = standardizeBoston NumericColumns rows
let model = fitPca table.Columns.Length table.X

## 主成分の数と累積寄与率

In [ ]:
let counts = [ 1 .. table.Columns.Length ]
let cumulative = model.ExplainedVarianceRatio |> Array.scan (+) 0.0 |> Array.tail

[
    Chart.Line(x = counts, y = cumulative, Name = "累積寄与率", ShowMarkers = true)
    Chart.Line(x = counts, y = List.replicate counts.Length 0.8, Name = "目安 0.8")
]
|> Chart.combine
|> Chart.withTitle "主成分の数と累積寄与率"
|> Chart.withXAxisStyle "主成分の数"
|> Chart.withYAxisStyle "累積寄与率"

In [ ]:
Array.zip (List.toArray counts) cumulative
|> Array.map (fun (count, ratio) -> {| 主成分の数 = count; 累積寄与率 = ratio |})

## 第 1・第 2 主成分の散布図

In [ ]:
let scores =
    Array.zip (transform model table.X) (rows |> List.map (fun row -> row.Crime) |> List.toArray)

scores
|> Array.groupBy snd
|> Array.map (fun (crime, group) ->
    Chart.Point(x = (group |> Array.map (fun (pc, _) -> pc[0])), y = (group |> Array.map (fun (pc, _) -> pc[1])), Name = crime))
|> Chart.combine
|> Chart.withTitle "第 1・第 2 主成分で見た地区"
|> Chart.withXAxisStyle "PC1"
|> Chart.withYAxisStyle "PC2"

In [ ]:
scores
|> Array.groupBy snd
|> Array.sortBy fst
|> Array.map (fun (crime, group) ->
    {|
        CRIME = crime
        PC1 = group |> Array.averageBy (fun (pc, _) -> pc[0])
        PC2 = group |> Array.averageBy (fun (pc, _) -> pc[1])
    |})

## 主成分への影響が大きい列

In [ ]:
[ 0; 1 ]
|> List.collect (fun i ->
    topLoadings 5 table.Columns model.Components[i]
    |> List.map (fun (column, value) -> {| 主成分 = $"第 {i + 1} 主成分"; 列 = column; 係数 = value |}))
|> List.toArray